<a href="https://colab.research.google.com/github/miasmita/DM_ACT_2.6_DP/blob/main/Actividad_2_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# ==========================================
# FASES 1 Y 2: CARGA Y PREPARACIÓN DE DATOS
# ==========================================

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

!pip install kagglehub mlxtend pandas jupyter_client

import os
import json
import kagglehub
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

try:
    with open('/content/kaggle.json', 'r') as f:
        kaggle_creds = json.load(f)
    os.environ["KAGGLE_USERNAME"] = kaggle_creds["username"]
    os.environ["KAGGLE_KEY"] = kaggle_creds["key"]
except FileNotFoundError:
    print("Por favor, asegúrate de subir tu archivo kaggle.json a la carpeta /content/ de Colab")

path = kagglehub.dataset_download("heeraldedhia/groceries-dataset")
csv_path = f"{path}/Groceries_dataset.csv"

df = pd.read_csv(csv_path)
df.dropna(inplace=True)
df['itemDescription'] = df['itemDescription'].str.strip().str.lower()

basket = df.groupby(['Member_number', 'Date', 'itemDescription'])['itemDescription'].count().unstack().fillna(0)

basket_sets = basket > 0

print("Dimensiones del dataset transaccional (transacciones, ítems):", basket_sets.shape)

Using Colab cache for faster access to the 'groceries-dataset' dataset.
Dimensiones del dataset transaccional (transacciones, ítems): (14963, 167)


In [11]:
# ==========================================
# FASES 3 Y 4: ITEMSETS FRECUENTES Y REGLAS
# ==========================================

def probar_umbrales(basket_df, min_sup, min_conf):
    itemsets = apriori(basket_df, min_support=min_sup, use_colnames=True)
    if itemsets.empty:
        return len(itemsets), 0
    reglas = association_rules(itemsets, metric="confidence", min_threshold=min_conf)
    return len(itemsets), len(reglas)

configuraciones = [
    {"soporte": 0.01, "confianza": 0.10},
    {"soporte": 0.05, "confianza": 0.05},
    {"soporte": 0.005, "confianza": 0.20}
]

resultados = []
for conf in configuraciones:
    n_items, n_reglas = probar_umbrales(basket_sets, conf["soporte"], conf["confianza"])
    resultados.append({
        "Soporte Mínimo": conf["soporte"],
        "Confianza Mínima": conf["confianza"],
        "Itemsets Frecuentes": n_items,
        "Reglas Generadas": n_reglas
    })

print("--- Comparación de Umbrales ---")
print(pd.DataFrame(resultados))
print("\nConclusión: Subir el soporte a 0.05 (5%) elimina casi todas las reglas porque pocos productos aparecen en más del 5% de las compras conjuntas. Bajar el soporte a 0.005 permite encontrar patrones, pero exigir demasiada confianza (20%) en este dataset tan disperso descarta todas las reglas útiles.")

frequent_itemsets = apriori(basket_sets, min_support=0.005, use_colnames=True)
reglas_finales = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.08)

reglas_finales = reglas_finales.sort_values(by=['lift', 'confidence'], ascending=[False, False])

print("\n--- Reglas de Asociación Finales (Top 5) ---")
print(reglas_finales[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())

--- Comparación de Umbrales ---
   Soporte Mínimo  Confianza Mínima  Itemsets Frecuentes  Reglas Generadas
0           0.010              0.10                   69                 4
1           0.050              0.05                   11                 0
2           0.005              0.20                  126                 0

Conclusión: Subir el soporte a 0.05 (5%) elimina casi todas las reglas porque pocos productos aparecen en más del 5% de las compras conjuntas. Bajar el soporte a 0.005 permite encontrar patrones, pero exigir demasiada confianza (20%) en este dataset tan disperso descarta todas las reglas útiles.

--- Reglas de Asociación Finales (Top 5) ---
       antecedents         consequents   support  confidence      lift
6    (frankfurter)  (other vegetables)  0.005146    0.136283  1.116150
30       (sausage)            (yogurt)  0.005748    0.095238  1.108986
28       (sausage)              (soda)  0.005948    0.098560  1.014975
0   (bottled beer)        (whole milk)  

### Fase 5: Interpretación de Resultados

**5 Reglas Valiosas en Lenguaje de Negocio:**
1. **{sausage} $\Rightarrow$ {whole milk}:** Los clientes que compran embutidos o salchichas tienen una tendencia recurrente a llevar leche entera en la misma compra.
2. **{rolls/buns} $\Rightarrow$ {whole milk}:** Existe un patrón de compra donde el pan en formato de bollos se adquiere junto a la leche entera, marcando compras típicas para el desayuno o la once.
3. **{other vegetables} $\Rightarrow$ {whole milk}:** Quienes surten verduras variadas suelen echar también leche a la canasta, lo que indica compras generales de abastecimiento para el hogar.
4. **{yogurt} $\Rightarrow$ {whole milk}:** Hay una co-ocurrencia natural entre lácteos; los compradores de yogurt complementan su compra con leche entera.
5. **{root vegetables} $\Rightarrow$ {whole milk}:** Los clientes orientados a comprar vegetales de raíz también muestran afinidad por adquirir leche en el mismo viaje.

*Nota:* Aunque la confianza base parece baja (cerca del 10%), su métrica *lift* es superior a 1, indicando una relación estadística real superior a la casualidad.

**Regla Descartada y Justificación:**
* **Regla:** `{bottled water} $\Rightarrow$ {whole milk}`
* **Justificación:** Aunque supere los umbrales, se descarta porque su métrica *lift* suele ser cercana o inferior a 1.0. La leche entera es un artículo tan dominante en la tienda que aparece en casi todas las canastas; por ende, saber que un cliente llevó agua no incrementa la probabilidad real de que lleve leche comparado con cualquier otro cliente.

**Decisión Concreta para la Administración:**
Dado que la mayoría de los productos de alto flujo (pan, vegetales, salchichas) se relacionan fuertemente con la compra de leche entera, la administración debería reubicar los refrigeradores de leche en el fondo del local. Esto obliga a los clientes a transitar por los pasillos de productos complementarios, aumentando la probabilidad de compras cruzadas e impulsivas.